### The following code is related to Figure 2D and Supplementary Figure 10A, which present the Spearman correlation between RNA expression and CLR-normalized protein levels. Furthermore, Gene Ontology biological process (GO-BP) analysis was performed using the top 200 most highly correlated genes.

In [ ]:
library(tibble)
library(stringr)
library(reshape2)
library(tidyr)
library(furrr)
library(future)
library(dplyr)
library(Seurat)
library(ggplot2)

In [ ]:
parallel::detectCores()

In [ ]:
setwd('/mnt/data/khm_scRNA/huh7_zxs/')
seurat_obj <- readRDS('./result_zxs/scdata_filter.rds')

In [ ]:
seurat_obj <- subset(seurat_obj,subset = batch %in% c('normal2','Tatin'))

In [ ]:
seurat_obj$batch %>% table()

In [ ]:
DefaultAssay(seurat_obj) <- 'Protein'
seurat_obj <- NormalizeData(object = seurat_obj,normalization.method = "CLR", margin =2)
seurat_obj

In [ ]:
DefaultAssay(seurat_obj) <- 'RNA'
seurat_obj

## Tatin

In [ ]:
seurat_obj$batch %>% unique()
seurat_obj <- subset(seurat_obj,subset = batch == 'Tatin')
meta_data <- seurat_obj@meta.data
meta_data %>% head()

In [ ]:
Idents(seurat_obj) <- seurat_obj$sgRNA_type
diff_gene <- FindAllMarkers(
    object = seurat_obj,
    logfc.threshold = 0.1,
    test.use = "wilcox"
)

In [ ]:
diff_gene %>% head()

In [ ]:
getwd()

In [ ]:
gene_select <- diff_gene %>% 
    filter(abs(avg_log2FC)>0.1,p_val<0.05) %>% 
    pull(gene) %>% unique()
gene_select %>% length()

In [ ]:
data_diffgene <- diff_gene %>% 
    filter(abs(avg_log2FC)>0.1,p_val<0.05) 
write.csv(data_diffgene,'./result_zxs/GO_result_Tatin/data_diffgene.csv')

In [ ]:
data_protein <- seurat_obj@assays$Protein@data %>% as.data.frame() %>% 
  rownames_to_column('protein') %>% 
  mutate(
    protein = protein %>% str_remove('-pAbO')
  ) %>% 
  pivot_longer(cols = names(.)[-1],names_to = 'cell_id',values_to = 'Exprossion')
data_protein$protein %>% table()

In [ ]:
data_protein %>% head()

In [ ]:
data_rna <- seurat_obj@assays$RNA$data %>% as.data.frame() %>% 
  filter(rownames(.) %in% gene_select) %>% 
  rownames_to_column('rna') %>% 
  pivot_longer(cols = names(.)[-1],names_to = 'cell_id',values_to = 'Exprossion')
data_rna$rna %>% unique() %>% length()

#### multi-core processing

In [ ]:
data_rna %>% head()

In [ ]:
plan(multisession, workers = 20)
plan()

In [ ]:
res_spearman <- future_map(data_protein$protein %>% unique(), function(protein_select){
  print(protein_select)
  
  proetin_select_exp <- data_protein %>% 
    filter(protein == protein_select)
  
  res_tmp <- lapply(data_rna$rna %>% unique(), function(rna_select){
    data_exp <- data_rna %>% 
      filter(rna == rna_select) %>% 
      left_join(proetin_select_exp, by = 'cell_id', suffix = c("_rna", "_protein"))
    
    tmp <- cor(data_exp$Exprossion_rna, data_exp$Exprossion_protein, method = 'spearman')
    return(tmp)
  })
  
  names(res_tmp) <- data_rna$rna %>% unique()
  
  data_res_rna <- res_tmp %>% stack() %>% 
    rename_all(~c('spearman_score','rna')) %>% 
    dplyr::select('rna','spearman_score')
  
  return(data_res_rna)
})
names(res_spearman) <- data_protein$protein %>% unique()

In [ ]:
res_spearman %>% length()
names(res_spearman)
res_spearman[[1]] %>% dim()
res_spearman[[1]] %>% head()

In [ ]:
getwd()

In [ ]:
res_spearman_tmp <- res_spearman %>% 
  purrr::imap_dfr(~mutate(.x, protein = .y))
res_spearman_tmp %>% head()
saveRDS(res_spearman,'./result_zxs/res_spearman_highdiff.rds')
saveRDS(res_spearman_tmp,'./result_zxs/res_spearman_tmp_highdiff.rds')

In [ ]:
plan()

In [ ]:
options(future.rng.onMisuse = "ignore")  
on.exit(future::plan('sequential'), add = TRUE)

In [ ]:
future::nbrOfWorkers()

### correlation check

In [ ]:
tmp <- lapply(X = names(res_spearman),FUN = function(x){
    print(x)
    tmp <- res_spearman[[x]]$spearman_score %>% .[!is.na(.)]
    paste(tmp %>% min(),tmp %>% max(),sep = ';') %>% print()
})

### Perform analysis on each gene iteratively

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)

In [ ]:
res_spearman <- readRDS('./result_zxs/res_spearman_highdiff.rds')
names(res_spearman)

In [ ]:
getwd()

In [ ]:
dir.create('./result_zxs/GO_result_Tatin',showWarnings = FALSE)

In [ ]:
# for(protein in names(res_spearman)){
go_res <- lapply(X = names(res_spearman),function(protein){
    print(protein)
    data_plot <- res_spearman[[protein]] %>% 
        filter(!is.na(spearman_score)) %>% 
        arrange(desc(spearman_score)) %>% 
        mutate(
            rank = 1:n()
        )
    options(repr.plot.height = 8,repr.plot.width = 6)
    p <- ggplot(data = data_plot,aes(x = rank,y = spearman_score)) +
        geom_point() +
        # geom_text_repel()
        labs(title = paste('Plot of',protein,sep = ' ')) +
        theme_bw() +
        theme(
            panel.grid = element_blank(),
            panel.border = element_blank(),
            axis.title = element_blank(),
            axis.line = element_line(colour = 'black'),
            axis.ticks = element_blank(),
            axis.text = element_text(size = 20),
            axis.text.x = element_text(angle = 30, hjust = 1),
            axis.text.y = element_text(hjust = 1),
            legend.text = element_text(size = 16),
            legend.position = 'right',
            legend.justification = c(0,1),
            legend.title = element_text(size = 20),
            strip.text = element_blank(),
            strip.background = element_blank()
        )
    print(p)
    gene_select <- data_plot$rna %>% .[1:200] %>% as.character()
    gene_select %>% head()
    gene_symbol <-
    bitr(
        gene_select,
        fromType = 'SYMBOL',
        toType = 'ENTREZID',
        OrgDb = "org.Hs.eg.db"
    )
    gene_symbol %>% head()
    enrich.go <- enrichGO(
      gene = gene_symbol$SYMBOL,
      OrgDb = 'org.Hs.eg.db',
      keyType = 'SYMBOL',
      ont = 'BP',
      pAdjustMethod = 'fdr',
      pvalueCutoff = 1,
      qvalueCutoff = 1,
      readable = FALSE
    )
    colnames(enrich.go@result)
    result <- enrich.go@result
    file_path <- paste('./result_zxs/GO_result_Tatin/GO_result_',protein,'.csv',sep = '')
    print(file_path)
    write.csv(result,file_path)
    return(list(p,enrich.go))
})

In [ ]:
length(go_res)
names(go_res)
names(go_res) <- names(res_spearman)
names(go_res)

In [ ]:
saveRDS(go_res,'./result_zxs/GO_result_Tatin/go_res.rds')

## Normal2

In [ ]:
setwd('/mnt/data/khm_scRNA/huh7_zxs/')
seurat_obj <- readRDS('./result_zxs/scdata_filter.rds')

In [ ]:
seurat_obj$batch %>% unique()
seurat_obj <- subset(seurat_obj,subset = batch == 'normal2')
meta_data <- seurat_obj@meta.data
meta_data %>% head()

In [ ]:
Idents(seurat_obj) <- seurat_obj$sgRNA_type
diff_gene <- FindAllMarkers(
    object = seurat_obj,
    logfc.threshold = 0.1,
    test.use = "wilcox"
)

In [ ]:
diff_gene %>% head()

In [ ]:
data_diffgene <- diff_gene %>% 
    filter(abs(avg_log2FC)>0.1,p_val<0.05) 
write.csv(data_diffgene,'./result_zxs/GO_result_normal/data_diffgene.csv')

In [ ]:
gene_select <- diff_gene %>% 
    filter(abs(avg_log2FC)>0.1,p_val<0.05) %>% 
    pull(gene) %>% unique()
gene_select %>% length()

In [ ]:
data_protein <- seurat_obj@assays$Protein@data %>% as.data.frame() %>% 
  rownames_to_column('protein') %>% 
  mutate(
    protein = protein %>% str_remove('-pAbO')
  ) %>% 
  pivot_longer(cols = names(.)[-1],names_to = 'cell_id',values_to = 'Exprossion')
data_protein$protein %>% table()

In [ ]:
data_rna <- seurat_obj@assays$RNA$data %>% as.data.frame() %>% 
  filter(rownames(.) %in% gene_select) %>% 
  rownames_to_column('rna') %>% 
  pivot_longer(cols = names(.)[-1],names_to = 'cell_id',values_to = 'Exprossion')
data_rna$rna %>% unique() %>% length()

#### multi-core processing

In [ ]:
data_rna %>% head()

In [ ]:
plan(multisession, workers = 10)
plan()

In [ ]:
res_spearman_normal <- future_map(data_protein$protein %>% unique(), function(protein_select){
  print(protein_select)
  
  proetin_select_exp <- data_protein %>% 
    filter(protein == protein_select)
  
  res_tmp <- lapply(data_rna$rna %>% unique(), function(rna_select){
    data_exp <- data_rna %>% 
      filter(rna == rna_select) %>% 
      left_join(proetin_select_exp, by = 'cell_id', suffix = c("_rna", "_protein"))
    
    tmp <- cor(data_exp$Exprossion_rna, data_exp$Exprossion_protein, method = 'spearman')
    return(tmp)
  })
  
  names(res_tmp) <- data_rna$rna %>% unique()
  
  data_res_rna <- res_tmp %>% stack() %>% 
    rename_all(~c('spearman_score','rna')) %>% 
    dplyr::select('rna','spearman_score')
  
  return(data_res_rna)
})
names(res_spearman_normal) <- data_protein$protein %>% unique()

In [ ]:
res_spearman_normal %>% length()
names(res_spearman_normal)
res_spearman_normal[[1]] %>% dim()
res_spearman_normal[[1]] %>% head()

In [ ]:
getwd()

In [ ]:
res_spearman_tmp_normal <- res_spearman_normal %>% 
  purrr::imap_dfr(~mutate(.x, protein = .y))
res_spearman_tmp_normal %>% head()
saveRDS(res_spearman_normal,'./result_zxs/res_spearman_highdiff_normal.rds')
saveRDS(res_spearman_tmp_normal,'./result_zxs/res_spearman_tmp_highdiff_normal.rds')

In [ ]:
res_spearman_tmp_normal %>% group_by(protein) %>% summarise(Counts = n())

### Correlation check

In [ ]:
tmp <- lapply(X = names(res_spearman_normal),FUN = function(x){
    print(x)
    tmp <- res_spearman_normal[[x]]$spearman_score %>% .[!is.na(.)]
    paste(tmp %>% min(),tmp %>% max(),sep = ';') %>% print()
})

### Perform analysis on each gene iteratively

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)

In [ ]:
res_spearman_normal <- readRDS('./result_zxs/res_spearman_highdiff_normal.rds')
names(res_spearman_normal)

In [ ]:
res_spearman_normal[[1]] %>% dim()

In [ ]:
getwd()

In [ ]:
dir.create('./result_zxs/GO_result_normal',showWarnings = FALSE)

In [ ]:
go_res <- lapply(X = names(res_spearman_normal),function(protein){
    print(protein)
    data_plot <- res_spearman_normal[[protein]] %>% 
        filter(!is.na(spearman_score)) %>% 
        arrange(desc(spearman_score)) %>% 
        mutate(
            rank = 1:n()
        )
    options(repr.plot.height = 8,repr.plot.width = 6)
    p <- ggplot(data = data_plot,aes(x = rank,y = spearman_score)) +
        geom_point() +
        # geom_text_repel()
        labs(title = paste('Plot of',protein,sep = ' ')) +
        theme_bw() +
        theme(
            panel.grid = element_blank(),
            panel.border = element_blank(),
            axis.title = element_blank(),
            axis.line = element_line(colour = 'black'),
            axis.ticks = element_blank(),
            axis.text = element_text(size = 20),
            axis.text.x = element_text(angle = 30, hjust = 1),
            axis.text.y = element_text(hjust = 1),
            legend.text = element_text(size = 16),
            legend.position = 'right',
            legend.justification = c(0,1),
            legend.title = element_text(size = 20),
            strip.text = element_blank(),
            strip.background = element_blank()
        )
    print(p)
    gene_select <- data_plot$rna %>% .[1:200] %>% as.character()
    gene_select %>% head()
    gene_symbol <-
    bitr(
        gene_select,
        fromType = 'SYMBOL',
        toType = 'ENTREZID',
        OrgDb = "org.Hs.eg.db"
    )
    gene_symbol %>% head()
    enrich.go <- enrichGO(
      gene = gene_symbol$SYMBOL,
      OrgDb = 'org.Hs.eg.db',
      keyType = 'SYMBOL',
      ont = 'BP',
      pAdjustMethod = 'fdr',
      pvalueCutoff = 1,
      qvalueCutoff = 1,
      readable = FALSE
    )
    colnames(enrich.go@result)
    result <- enrich.go@result
    file_path <- paste('./result_zxs/GO_result_normal/GO_result_',protein,'.csv',sep = '')
    print(file_path)
    write.csv(result,file_path)
    return(list(p,enrich.go))
})

In [ ]:
length(go_res)
names(go_res)
names(go_res) <- names(res_spearman_normal)
names(go_res)

In [ ]:
saveRDS(go_res,'./result_zxs/GO_result_normal/go_res.rds')

## plot

In [ ]:
setwd('/mnt/data/khm_scRNA/huh7_zxs/')
getwd()
dir()

In [ ]:
getwd()

### Normal

In [ ]:
res_spearman <- readRDS('./result_zxs/res_spearman_highdiff_normal.rds')
names(res_spearman)

In [ ]:
gene_list_normal <- list(
  ALOD4 = c("AFP", "HMGCR", "FADS1", "EBP", "MVD", "GPX4", "COQ5", "ECH1", "VDAC1", "LRP1", "VAPB"),
  `p-RPS6` = c("RPS6", "RPS15", "PLK1", "SKP2", "EIF4A3", "AURKB", "PGAM1", "MAZ", "RPL37A", "RPL39", "LAMTOR3"),
  pAKT = c("RPS6", "VEGFA", "BMP2", "PDGFRA", "APOC1", "PRKAR2A", "LYN", "FGF14", "SKP2", "PPP2CB"),
  pP65 = c("IL6ST", "IL1R1", "LYN", "VEGFA", "CXCL12", "SERPINE2"),
  `c-MYc` = c("KPNB1", "RPS6", "RFC4", "HDGF", "UCHL1", "SKP2")
)


In [ ]:
sgRNAIdentity <- 'pP65'
data_plot <- res_spearman[[sgRNAIdentity]] %>% 
        filter(!is.na(spearman_score)) %>% 
        arrange(desc(spearman_score)) %>% 
        mutate(
            rank = 1:n()
        )

tmp <- gene_list_normal[[sgRNAIdentity]]
tmp

In [ ]:
tmp[!(tmp %in% data_plot$rna)]
tmp[(tmp %in% data_plot$rna)]

In [ ]:
save_path <- './result_figs/protein_mRNA_spearman'
dir.create(save_path,showWarnings = FALSE)
dir(save_path)

In [ ]:
color_use <- '#5B3794'
lapply(names(gene_list_normal),function(sgRNAIdentity){
    print(sgRNAIdentity)
    gene_label <- gene_list_normal[[sgRNAIdentity]]
    data_plot <- res_spearman[[sgRNAIdentity]] %>% 
        filter(!is.na(spearman_score)) %>% 
        arrange(desc(spearman_score)) %>% 
        mutate(
            rank = 1:n()
        )
    data_plot_label <- data_plot %>% 
      filter(rna %in% gene_label) %>%
      mutate(
          nudge_x_val = data_plot$rank %>% max()*0.35,
          nudge_y_val = data_plot$spearman_score %>% max()  %>% {.*0.2}
      )
    options(repr.plot.height = 8,repr.plot.width = 7)
    p <- ggplot(data = data_plot,aes(x = rank,y = spearman_score)) +
        geom_point() +
        geom_text_repel(
            data = data_plot_label,
            aes(x = rank,y = spearman_score,label = rna),force = 100,max.time = 20,
            arrow = arrow(length = unit(0.02, "npc"), type = "closed", ends = "first"),
            color = color_use,nudge_x = data_plot_label$nudge_x_val,nudge_y = data_plot_label$nudge_y_val,
            segment.curvature = 0
        ) +
        geom_point(
            data = data_plot %>% filter(rna %in% gene_label),
            aes(x = rank,y = spearman_score),color = color_use,size = 3
        ) +
        annotate(
            "segment",
            x = -Inf,
            xend = 1.1 * max(data_plot$rank),
            y = 0,
            yend = 0,  
            color = color_use,linetype = "longdash"
        ) +
        annotate(
            "segment",
            x = 1.1 * max(data_plot$rank),
            xend = 1.1 * max(data_plot$rank),
            y = 0.02,
            yend = max(data_plot$spearman_score)*0.8, 
            arrow = arrow(length = unit(0.2, "cm"), type = "closed"),
            color = color_use
        ) +
        annotate(
            "text",
            x = 1.1 * max(data_plot$rank) + 10,
            y = max(data_plot$spearman_score)*0.4+0.01,
            label = paste("mRNA Up in\nsg-",sgRNAIdentity,sep = ''),
            hjust = -0.1,size = 6,color = color_use
        ) +
        annotate(
            "segment",
            x = 1.1 * max(data_plot$rank),
            xend = 1.1 * max(data_plot$rank),
            y = -0.02,
            yend = min(data_plot$spearman_score)*0.8,  
            arrow = arrow(length = unit(0.2, "cm"), type = "closed"),
            color = "black"
        ) +
        annotate(
            "text",
            x = 1.1 * max(data_plot$rank) + 10,
            y = min(data_plot$spearman_score)*0.4-0.01,
            label = paste("mRNA Down in\nsg-",sgRNAIdentity,sep = ''),
            hjust = -0.1,size = 6,color = 'black'
        ) +
        coord_cartesian(clip = "off") +
        labs(x = 'Genes',y = paste(sgRNAIdentity,'r'),title = paste(sgRNAIdentity,'Regulated RNA')) +
        theme_bw() +
        theme(
            plot.margin = margin(5.5, 120, 5.5, 5.5),
            panel.grid = element_blank(),
            panel.border = element_blank(),
            plot.title = element_text(size = 24,hjust = 0.5),
            axis.title = element_blank(),
            axis.line = element_line(colour = 'black'),
            axis.ticks = element_blank(),
            axis.text = element_text(size = 20),
            # axis.text.x = element_text(angle = 30, hjust = 1),
            axis.text.x = element_blank(),
            axis.text.y = element_text(hjust = 1),
            legend.text = element_text(size = 16),
            legend.position = 'right',
            legend.justification = c(0,1),
            legend.title = element_text(size = 20),
            strip.text = element_blank(),
            strip.background = element_blank()
        )
    p
    file_path <- paste(save_path,'/Normal_',sgRNAIdentity,'.pdf',sep = '')
    ggsave(plot = p,filename = file_path,width = 7,height = 8)
    p
})

### Tatin

In [ ]:
res_spearman <- readRDS('./result_zxs/res_spearman_highdiff.rds')
names(res_spearman)

In [ ]:
gene_list_tatin <- list(
  ALOD4 = c("SREBF1", "ABCA1", "ABCG1", "GPX4", "FTL"),
  `p-RPS6` = c("RPS5", "RPS15", "EIF4EBP1", "SHC1", "PLK1", "ATP5F1A"),
  pAKT = c("SREBF1", "SHC1", "GRB2", "SQSTM1", "ITGB1"),
  pP65 = c("IL6ST", "SHC1", "ITGB1", "CXCL5", "SERPINE1", "SQSTM1", "GPX4"),
  `c-MYc` = c("NAP1L1", "RCL1", "SMARCC1", "ATIC", "GNAS")
)

In [ ]:
sgRNAIdentity <- 'pP65'
data_plot <- res_spearman[[sgRNAIdentity]] %>% 
        filter(!is.na(spearman_score)) %>% 
        arrange(desc(spearman_score)) %>% 
        mutate(
            rank = 1:n()
        )

tmp <- gene_list_tatin[[sgRNAIdentity]]
tmp

In [ ]:
tmp[!(tmp %in% data_plot$rna)]
tmp[(tmp %in% data_plot$rna)]

In [ ]:
save_path <- './result_figs/protein_mRNA_spearman'
dir.create(save_path,showWarnings = FALSE)
dir(save_path)

In [ ]:
color_use <- '#5B3794'
lapply(names(gene_list_tatin),function(sgRNAIdentity){
    print(sgRNAIdentity)
    gene_label <- gene_list_tatin[[sgRNAIdentity]]
    data_plot <- res_spearman[[sgRNAIdentity]] %>% 
        filter(!is.na(spearman_score)) %>% 
        arrange(desc(spearman_score)) %>% 
        mutate(
            rank = 1:n()
        )
    data_plot_label <- data_plot %>% 
      filter(rna %in% gene_label) %>%
      mutate(
          nudge_x_val = data_plot$rank %>% max()*0.35,
          nudge_y_val = data_plot$spearman_score %>% max()  %>% {.*0.2}
      )
    options(repr.plot.height = 8,repr.plot.width = 7)
    p <- ggplot(data = data_plot,aes(x = rank,y = spearman_score)) +
        geom_point() +
        geom_text_repel(
            data = data_plot_label,
            aes(x = rank,y = spearman_score,label = rna),force = 100,max.time = 20,
            arrow = arrow(length = unit(0.02, "npc"), type = "closed", ends = "first"),
            color = color_use,nudge_x = data_plot_label$nudge_x_val,nudge_y = data_plot_label$nudge_y_val,
            segment.curvature = 0
        ) +
        geom_point(
            data = data_plot %>% filter(rna %in% gene_label),
            aes(x = rank,y = spearman_score),color = color_use,size = 3
        ) +
        annotate(
            "segment",
            x = -Inf,
            xend = 1.1 * max(data_plot$rank),
            y = 0,
            yend = 0,  
            color = color_use,linetype = "longdash"
        ) +
        annotate(
            "segment",
            x = 1.1 * max(data_plot$rank),
            xend = 1.1 * max(data_plot$rank),
            y = 0.02,
            yend = max(data_plot$spearman_score)*0.8,  
            arrow = arrow(length = unit(0.2, "cm"), type = "closed"),
            color = color_use
        ) +
        annotate(
            "text",
            x = 1.1 * max(data_plot$rank) + 10,
            y = max(data_plot$spearman_score)*0.4+0.01,
            label = paste("mRNA Up in\nsg-",sgRNAIdentity,sep = ''),
            hjust = -0.1,size = 6,color = color_use
        ) +
        annotate(
            "segment",
            x = 1.1 * max(data_plot$rank),
            xend = 1.1 * max(data_plot$rank),
            y = -0.02,
            yend = min(data_plot$spearman_score)*0.8,  
            arrow = arrow(length = unit(0.2, "cm"), type = "closed"),
            color = "black"
        ) +
        annotate(
            "text",
            x = 1.1 * max(data_plot$rank) + 10,
            y = min(data_plot$spearman_score)*0.4-0.01,
            label = paste("mRNA Down in\nsg-",sgRNAIdentity,sep = ''),
            hjust = -0.1,size = 6,color = 'black'
        ) +
        coord_cartesian(clip = "off") +
        labs(x = 'Genes',y = paste(sgRNAIdentity,'r'),title = paste(sgRNAIdentity,'Regulated RNA')) +
        theme_bw() +
        theme(
            plot.margin = margin(5.5, 120, 5.5, 5.5),
            panel.grid = element_blank(),
            panel.border = element_blank(),
            plot.title = element_text(size = 24,hjust = 0.5),
            axis.title = element_blank(),
            axis.line = element_line(colour = 'black'),
            axis.ticks = element_blank(),
            axis.text = element_text(size = 20),
            # axis.text.x = element_text(angle = 30, hjust = 1),
            axis.text.x = element_blank(),
            axis.text.y = element_text(hjust = 1),
            legend.text = element_text(size = 16),
            legend.position = 'right',
            legend.justification = c(0,1),
            legend.title = element_text(size = 20),
            strip.text = element_blank(),
            strip.background = element_blank()
        )
    p
    file_path <- paste(save_path,'/Tatin_',sgRNAIdentity,'.pdf',sep = '')
    ggsave(plot = p,filename = file_path,width = 7,height = 8)
    p
})

### GO

In [ ]:
library(clusterProfiler)
library(org.Hs.eg.db)
getwd()

In [ ]:
go_res <- readRDS('./result_zxs/GO_result_normal/go_res.rds')

In [ ]:
go_res %>% names()

In [ ]:
go_list <- list(
  'p-RPS6' = c("cellular respiration", "cytoplasmic translation", "detoxification"),
  pAKT = c("response to toxic substance", "extracellular matrix organization", "cellular response to toxic substance")
)

In [ ]:
custom_palette <- colorRampPalette(c("#F5F4F8", "#B8B3D4"))
custom_palette(3)

In [ ]:
lapply(names(go_list),function(protein_select){
    print(protein_select)
    data_plot <- go_res[[protein_select]][[2]]@result %>% 
        filter(Description %in% go_list[[protein_select]]) %>% 
        arrange(RichFactor) %>% 
        mutate(Description = factor(Description,levels = Description %>% unique()))
    min_value <- data_plot$p.adjust %>% {-log10(.) * 10 } %>% min() %>% ceiling() %>% {./10}
    max_value <- data_plot$p.adjust %>% {-log10(.) * 10 } %>% max() %>% floor() %>% {./10}
    options(repr.plot.height = 5,repr.plot.width = 8)
    p <- ggplot(data_plot, aes(x = RichFactor, y = Description, fill = -log10(p.adjust))) +
      geom_bar(stat = 'identity',width = 0.5) +
      geom_text(aes(x = 0.01, y = Description,label = str_replace(Description, "^\\s+", "") %>% str_to_title()),size = 7,angle = 0,hjust = 0.3,vjust = -3,color = 'black') +
      scale_fill_gradient2(
            name = '-Log10(p.adjust)',
            low = custom_palette(3)[1],mid = custom_palette(3)[2],high = custom_palette(3)[3],
            midpoint = (min_value + max_value)/2,
          breaks = c(min_value,(min_value + max_value)/2,max_value),
          labels = c(min_value,(min_value + max_value)/2,max_value)
        ) +
      guides(fill = guide_colorbar(title.position = "left", title.theme = element_text(angle = 90,vjust = 0.5))) +
      scale_x_continuous(expand = c(0.01,0)) +
      scale_y_discrete(expand = c(0,0.4)) +
      labs(y = 'Pathway',x = 'RichFactor') +
      theme_classic(base_size = 20) +
      coord_cartesian(clip = "off") +
      theme(
        plot.margin = unit(c(60, 0, 0, 0), "pt"),
        plot.title = element_text(hjust = 0.5,size = 32),
        legend.title = element_text(size = 24,hjust = 0),
        legend.text = element_text(size = 20),
        legend.key.height = unit(1.4, "cm"),
        legend.key.width = unit(1.2, "cm"),
        axis.text.x = element_blank(),
        axis.text.y = element_blank(),
        axis.title.x = element_blank(),
        strip.background = element_blank(),
        strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
        strip.placement = 'outside'
      )
    file_path <- paste('./result_figs/GO_',protein_select,'.pdf',sep = '')
    ggsave(filename = file_path,plot = p,width = 8,height = 5)
})

In [ ]:
getwd()

#### single sgIdentity

In [ ]:
go_list[[protein_select]]

In [ ]:
protein_select <- names(go_list)[1]
print(protein_select)
data_plot <- go_res[[protein_select]][[2]]@result %>% 
    dplyr::filter(Description %in% go_list[[protein_select]]) %>% 
    arrange(RichFactor) %>% 
    mutate(Description = factor(Description,levels = Description %>% unique()))
data_plot %>% head()

In [ ]:
min_value <- data_plot$p.adjust %>% {-log10(.) * 10 } %>% min() %>% ceiling() %>% {./10}
max_value <- data_plot$p.adjust %>% {-log10(.) * 10 } %>% max() %>% floor() %>% {./10}
(min_value + max_value)/2

In [ ]:
options(repr.plot.height = 5,repr.plot.width = 8)
ggplot(data_plot, aes(x = RichFactor, y = Description, fill = -log10(p.adjust))) +
  geom_bar(stat = 'identity',width = 0.5) +
  geom_text(aes(x = 0.01, y = Description,label = str_replace(Description, "^\\s+", "") %>% str_to_title()),size = 7,angle = 0,hjust = 0.3,vjust = -3,color = 'black') +
  scale_fill_gradient2(
        name = '-Log10(p.adjust)',
        low = custom_palette(3)[1],mid = custom_palette(3)[2],high = custom_palette(3)[3],
        midpoint = (min_value + max_value)/2,
      breaks = c(min_value,(min_value + max_value)/2,max_value),
      labels = c(min_value,(min_value + max_value)/2,max_value)
    ) +
  guides(fill = guide_colorbar(title.position = "left", title.theme = element_text(angle = 90,vjust = 0.5))) +
  scale_x_continuous(expand = c(0.01,0)) +
  scale_y_discrete(expand = c(0,0.4)) +
  labs(y = 'Pathway',x = 'RichFactor') +
  theme_classic(base_size = 20) +
  coord_cartesian(clip = "off") +
  theme(
    plot.margin = unit(c(60, 0, 0, 0), "pt"),
    plot.title = element_text(hjust = 0.5,size = 32),
    legend.title = element_text(size = 24,hjust = 0),
    legend.text = element_text(size = 20),
    legend.key.height = unit(1.4, "cm"),
    legend.key.width = unit(1.2, "cm"),
    axis.text.x = element_blank(),
    axis.text.y = element_blank(),
    axis.title.x = element_blank(),
    strip.background = element_blank(),
    strip.text = element_text(size = 20,face = 'italic',vjust = 0.7),
    strip.placement = 'outside'
  )

## endline